# USER TEST — Area Coverage · kandidat 0.6.4rc1

Notebook ini adalah kandidat lokal untuk pengujian owner. Tersedia pada branch pengujian fix/area-user-testing; belum dipublikasikan ke Kaggle atau dipromosikan ke branch konsolidasi, dan belum dianggap stabil. HOME/RESULT tetap tersedia; mode awal **Area Coverage**. Startup tidak menjalankan engine. Gunakan sesi baru dengan GPU, Internet, serta Secret **LocatioIntelligent** aktif. Jangan menempelkan isi credential ke notebook atau chat.

## Langkah pengujian

1. Run All, buka **GRADIO_PUBLIC_URL**, isi KDTK 4karakter dan Nama Toko, periksa Latitude/Longitude. Periode otomatis tanggal Jakarta. Default koordinat Sorong; sesuaikan jika lokasi berbeda.
2. Pada Action, pastikan **Area Coverage**. Tidak perlu upload video atau menjalankan Traffic/PPAC. Klik pemeriksaan kesiapan, kemudian **Process**.
3. Di RESULT, periksa status Area dan tabel alokasi, kompetitor/Indomaret, serta peta interaktif Folium/Esri. Coba pan/zoom dan layer. PPAC **NOT_RUN** pada Area-only adalah wajar.
4. Catat job_id yang dihasilkan. Klik **Buka hasil BQ** untuk membaca kembali job yang sama dari penyimpanan; jangan hanya menilai hasil yang masih ada di RAM.
5. Jika gagal, klik **Periksa penyebab kegagalan Area** lalu **Siapkan paket diagnosis Area**. Unduh ZIP, catat tahap/kode error dan job_id. Tidak perlu menjalankan Traffic untuk menyelidiki Area.
6. Berikan hasil pengujian dan paket diagnosis. Push ke refactor/deployment-foundation serta pembaruan dokumen aktif menunggu hasil owner. CI lokal bukan bukti fresh Area Sorong atau IAM berhasil.

Output menggunakan gs://aretfak-location-intelligence dan dataset locdeproject.location_intelligence. Tabel harus sudah disiapkan; notebook tidak membuat atau menghapus tabel. Source model/PBF/reference tetap terpisah. Jejak tahapan disimpan lokal per run pada area/area-trace.json dan ikut manifest GCS bila publikasi berhasil. ZIP diagnosis memuat metadata error terbatas, bukan private key/video/source rows. Jika publikasi gagal, hasil dan diagnosis lokal tetap dapat diunduh pada sesi ini.

Setelah selesai, interrupt cell server. Bila UI tidak bisa mengunduh ZIP, jalankan cell diagnosis terakhir lalu ambil file dari panel Files Kaggle. Jangan mematikan seluruh sesi sebelum menyimpan bukti.


In [ ]:
import base64,hashlib,io,zipfile,json,sys,subprocess,importlib,importlib.metadata
from pathlib import Path
BASE=Path('/kaggle/working/gate2-area-user-test');BASE.mkdir(parents=True,exist_ok=True)
PAYLOAD=base64.b64decode('...')
assert hashlib.sha256(PAYLOAD).hexdigest()=='e5f63a05443caad7d8438ad0a248e25ef99a9f88bdd9503342c702d620fefe47'
zipfile.ZipFile(io.BytesIO(PAYLOAD)).extractall(BASE)
WHEEL=BASE/'package'/'traffic_location_engines-0.6.4rc1-py3-none-any.whl'
assert hashlib.sha256(WHEEL.read_bytes()).hexdigest()=='bc1583696aef7d4387766b497b863a9c185134d287c198491c0f369e2c6b5c3d'
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(WHEEL)],check=True)
subprocess.run([sys.executable,'-m','pip','install','gradio==5.49.1','google-cloud-storage==3.6.0','google-cloud-bigquery==3.46.2','google-crc32c==1.7.1','jsonschema==4.26.0','numpy==2.1.3','pandas==2.3.3'],check=True)
for key in list(sys.modules):
    if key=='traffic_location' or key.startswith('traffic_location.'):del sys.modules[key]
importlib.invalidate_caches()
import traffic_location,gradio
assert gradio.__version__=='5.49.1','Restart sesi agar Gradio sesuai versi terpasang.'
package_root=Path(traffic_location.__file__).parent
with zipfile.ZipFile(WHEEL) as package:
    for name in package.namelist():
        if name.startswith('traffic_location/') and name.endswith('.py'):
            assert (package_root/Path(name).relative_to('traffic_location')).read_bytes()==package.read(name),name
print('Paket dan UI siap. Belum ada engine berjalan.')


In [ ]:
from traffic_location.integration.gradio_app import launch
if 'APP' in globals():
    APP.close()
APP, PUBLIC_URL=launch(BASE/'system',BASE/'config',share=True,default_mode="Area Coverage")
print('GRADIO_PUBLIC_URL='+PUBLIC_URL,flush=True)
print('Buka URL publik tersebut. Engine berjalan setelah tombol Mulai proses ditekan.',flush=True)
try:
    APP.block_thread()
finally:
    APP.close()


## Diagnosis setelah menghentikan server (opsional)
Cell ini hanya membaca jejak run Area terakhir pada workspace notebook ini. Tidak menjalankan engine atau menghubungi cloud.

In [ ]:
from traffic_location.integration.area_trace import export_report
from IPython.display import display,FileLink
traces=sorted((BASE/'system/targets').glob('*/runs/*/area/area-trace.json'),key=lambda p:p.stat().st_mtime)
if traces:
    workspace=traces[-1].parents[3]
    diagnostic=export_report(workspace,BASE/'diagnostics/area-diagnostics.zip')
    print('DIAGNOSTIC_ZIP='+str(diagnostic))
    display(FileLink(str(diagnostic)))
else:
    print('Belum ada run Area pada workspace ini. Catat pesan startup/validasi jika UI belum bisa menjalankan Area.')
